In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

In [5]:
!pip install langchain chromadb
!pip install langchain-chroma
!pip install langchain_huggingface
!pip install langchain-community
!pip install sentence-transformers

In [4]:
df = pd.read_csv('BBCNews.csv')
df.columns = ['News_ID','Description','Tags']
df.head()

,News_ID,Description,Tags
0,0,chelsea sack mutu chelsea have sacked adrian ...,"sports, stamford bridge, football association,..."
1,1,record fails to lift lacklustre meet yelena i...,"sports, madrid, birmingham, france, scotland, ..."
2,2,edu describes tunnel fracas arsenals edu has ...,"sports, derby, brazil, tunnel fracasedu, food,..."
3,3,ogara revels in ireland victory ireland flyha...,"sports, bbc, united kingdom, ireland, brian o'..."
4,4,unclear future for striker baros liverpool fo...,"sports, liverpool, daily sport, millennium sta..."


In [5]:
df.shape

(2410, 3)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2410 entries, 0 to 2409
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   News_ID      2410 non-null   int64 
 1   Description  2410 non-null   object
 2   Tags         2410 non-null   object
dtypes: int64(1), object(2)
memory usage: 56.6+ KB


In [7]:
df = df.dropna(subset=["Description", "Tags"])

In [8]:
df['Description'] = df['Description'].str.replace(r"\s+", " ", regex=True).str.strip()
df['Tags'] = df['Tags'].str.replace(r"\s+", " ", regex=True).str.strip()

In [9]:
len(df['Description'].unique())

2126

In [10]:
len(df['Tags'].unique())

2167

In [11]:
duplicates = df[df["Description"].duplicated(keep=False)].sort_values("Description")

# duplicates.to_excel("duplicate_articles1.xlsx", index=False)

In [12]:
df1 = df.drop_duplicates(subset=["Description", "Tags"]).reset_index(drop=True)
before = len(df)
after = len(df1)

print("Rows removed:", before - after)
print("Rows remaining:", after)

Rows removed: 220
Rows remaining: 2190


In [13]:
def aggregate_tags(tags):
    unique_tags = []
    for tag_string in tags:
        for tag in tag_string.split(','):
            tag = tag.strip()
            if tag not in unique_tags:
                unique_tags.append(tag)

    return ", ".join(unique_tags)

df2 = (
    df1.groupby("Description", as_index=False)
      .agg({'Tags': aggregate_tags})
)
before = len(df1)
after = len(df2)

print("Rows removed:", before - after)
print("Rows remaining:", after)

Rows removed: 64
Rows remaining: 2126


In [14]:
from langchain_core.documents import Document

# Prepare documents for ChromaDB
documents = []
for _, row in df2.iterrows():
    content = row['Description']
    tags = row['Tags']
    meta = {"tags": tags}
    documents.append(Document(page_content=content, metadata=meta))

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings

hg_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

d:\AI Projects\RAG Practice\BBC_News_Extraction\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2373.73it/s]


In [15]:
from langchain_chroma import Chroma

current_directory = Path.cwd()
persist_directory = f"{current_directory}\chroma_db"

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=hg_embeddings,
    collection_name="bbc_news",
    persist_directory=persist_directory,
    collection_metadata={"hnsw:space": "ip"}
)

In [16]:
print(vectorstore._collection.count())

2126


In [18]:
results = vectorstore.similarity_search(
    "latest technology news",
    k=3
)
results

[Document(id='8302d7a8-1e19-49cc-b1e7-f4bb046c6e92', metadata={'tags': "entertainment, technology, bangkok, hewlett packard, nokia, the hp, mtv asia, microsoft, asia, europe, united states, portable media players, technology industry, digital media, technology giant, media hubs, hp's digital entertainment centre, carly fiorina, gwen stefani, bill gates, singer, chief of technology giant, business, economy of the united states, consumer electronics, american people of german descent, hewlett-packard, digital media player, consumer electronics show, digital camera, digital video recorder, digital revolution, digital cameras, mp3, digital video, computing"}, page_content='more power to the people says hp the digital revolution is focused on letting people tell and share their own stories according to carly fiorina chief of technology giant hewlett packard the job of firms such as hp now she said in a speech at the consumer electronics show ces was to ensure digital and physical worlds ful

In [19]:
for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("---")

more power to the people says hp the digital revolution is focused on letting people tell and share their own stories according to carly fiorina chief of technology giant hewlett packard the job of firms such as hp now she said in a speech at the consumer electronics show ces was to ensure digital and physical worlds fully converged she said the goal for was to make people the centre of technology ces showcases new gadgets that will be hitting the shelves in the techfest the largest of its kind in the world runs from to january the digital revolution is about the democratisation of technology and the experiences it makes possible she told delegates revolution has always been about giving power to the people she added the real story of the digital revolution is not just new products but the millions of experiences made possible and stories that millions can tell part of giving people more control has been about the freeing up of content such as images video and music crucial to this has